In [1]:
import os
import pwd
import numpy as np
import pandas as pd
import sys
from pyspark.sql import Window


from pyspark.sql import SparkSession
from random import randrange
import pyspark.sql.functions as F


username = pwd.getpwuid(os.getuid()).pw_name
hadoopFS=os.getenv('HADOOP_FS', None)
groupName = "H1"

print(os.getenv('SPARK_HOME'))
print(f"hadoopFSs={hadoopFS}")
print(f"username={username}")
print(f"group={groupName}")

/opt/spark
hadoopFSs=hdfs://iccluster061.iccluster.epfl.ch:9000
username=hgruber
group=H1


In [2]:
spark = (SparkSession\
            .builder
            .appName(username + '-assignment-2')
            .config('spark.ui.port', randrange(4050, 4450, 5))
            .config("spark.executorEnv.PYTHONPATH", ":".join(sys.path))
            .config('spark.jars',
                    f'{hadoopFS}/data/com-490/jars/iceberg-spark-runtime-3.5_2.13-1.6.1.jar,'
                    f'{hadoopFS}/data/com-490/jars/sedona-spark-shaded-3.5_2.13-1.7.1.jar,'
                    f'{hadoopFS}/data/com-490/jars/geotools-wrapper-1.7.1-28.5.jar'
            )
            .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
            .config('spark.sql.catalog.iceberg', 'org.apache.iceberg.spark.SparkCatalog')
            .config('spark.sql.catalog.iceberg.type', 'hadoop')
            .config('spark.sql.catalog.iceberg.warehouse', f'{hadoopFS}/data/com-490/silver/')
            .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')
            .config('spark.sql.catalog.spark_catalog.type', 'hadoop')
            .config('spark.sql.catalog.spark_catalog.warehouse', f'{hadoopFS}/user/{username}/assignment-3/warehouse')
            .config("spark.sql.warehouse.dir", f'{hadoopFS}/user/{username}/assignment-3/spark/warehouse')
            .config("spark.executor.memory", "6g")
            .config("spark.executor.cores", "4")
            .config("spark.executor.instances", "4")
        ).master('yarn').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/14 11:01:18 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
spark.sparkContext

<SparkContext master=yarn appName=hgruber-assignment-2>

In [4]:
df_agency = spark.read.options(header=True).csv(
    "/data/com-490/bronze/sbb/agency/year=2026/month=01/day=31/agency.txt"
)

df_agency.createOrReplaceTempView("agency")

df_transfers = spark.read.csv(
    "/data/com-490/bronze/sbb/transfers/year=2026/month=01/day=31/transfers.txt", 
    header=True, 
    inferSchema=True
)

df_transfers.createOrReplaceTempView("transfers")

In [69]:
from sedona.spark import SedonaContext
spark = SedonaContext.create(spark)

operator_mapping = spark.table("iceberg.sbb.istdaten") \
    .select("operator_abrv", "operator_id", "operator_name") \
    .distinct()

operator_mapping.createOrReplaceTempView("op_map")

region_names = ["Lausanne", "Belmont-sur-Lausanne", "Le Mont-sur-Lausanne", "Romanel-sur-Lausanne", "Cheseaux-sur-Lausanne", "Ouest lausannois"]
formatted_regions = ", ".join([f"'{name}'" for name in region_names])

trip_stop_events_df = spark.sql(f"""
SELECT DISTINCT
 st.trip_id, st.stop_id, split_part(s.stop_id, ':', 1) AS cleaned_stop_id, s.stop_name, s.stop_lat, s.stop_lon, s.location_type, st.arrival_time, st.departure_time,
 st.stop_sequence, t.service_id, t.trip_short_name, ist.operator_id, ist.operator_name,
 c.monday, c.tuesday, c.wednesday, c.thursday, c.friday, c.saturday, c.sunday,
 c.start_date, c.end_date, r.route_short_name, 
 CASE 
        WHEN r.route_desc = 'EN' THEN 'NJ'
        WHEN r.route_desc IN ('EXB', 'KB', 'RUB', 'TX') THEN 'Bus'
        WHEN r.route_desc IN ('BP', 'FAE') THEN 'BAT'
        WHEN r.route_desc IN ('GB', 'PB', 'SL') THEN 'T'
        WHEN r.route_desc IN ('FUN', 'ASC') THEN 'M'
        WHEN r.route_desc IN ('ZUG', 'EST', 'ARZ', 'IRE') THEN 'R'
        ELSE r.route_desc 
    END AS transport_clean
 
FROM iceberg.sbb.stop_times st
 INNER JOIN iceberg.sbb.stops s 
    ON s.stop_id = st.stop_id AND s.pub_date = '2026-01-31' 
INNER JOIN iceberg.sbb.trips t 
    ON st.trip_id = t.trip_id AND t.pub_date = '2026-01-31'
INNER JOIN iceberg.sbb.calendar c 
    ON c.service_id = t.service_id AND c.pub_date = '2026-01-31'
INNER JOIN iceberg.sbb.routes r 
    ON r.route_id = t.route_id AND r.pub_date = '2026-01-31'
INNER JOIN agency a 
    ON a.agency_id = r.agency_id
 LEFT JOIN op_map ist 
    ON a.agency_name = ist.operator_name
 JOIN iceberg.geo.shapes g
    ON ST_Contains(ST_GeomFromWKB(g.wkb_geometry), ST_Point(s.stop_lon, s.stop_lat))

WHERE 
 st.pub_date = '2026-01-31' AND
 st.stop_id LIKE '85%' AND 
 g.name IN ({formatted_regions}) AND 
 g.level = 'district'
""")

trip_stop_events_df.cache()
row_count = trip_stop_events_df.count()
print(f"Data solidified in memory. Row count: {row_count}")
trip_stop_events_df.show(5, truncate=False)

output_path = "/user/groups/com-490/H1/final/v1/trip_stop_events.parquet"

trip_stop_events_df.write \
    .mode("overwrite") \
    .parquet(output_path)

26/05/14 14:15:23 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
26/05/14 14:15:23 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
26/05/14 14:15:23 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
26/05/14 14:15:23 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.geometryObjects.Geography, which is already registered.
26/05/14 14:15:23 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
26/05/14 14:15:23 WARN SimpleFunctionRegistry: The function st_envelope_aggr replaced a previously registered function.
26/05/14 14:15:23 WARN SimpleFunctionRegistry: The function st_intersection_aggr replaced a previously registered function.
26/05/14 14:15:23 WARN SimpleFunctionRegistry: The function st_union_aggr replaced a previously

Data solidified in memory. Row count: 680152
+------------------------+---------------+-----------------------+---------------+--------------+-------------+------------+--------------+-------------+----------+---------------+-----------+----------------------------------------------+------+-------+---------+--------+------+--------+------+----------+----------+----------------+---------------+
|trip_id                 |stop_id        |stop_name              |stop_lat       |stop_lon      |location_type|arrival_time|departure_time|stop_sequence|service_id|trip_short_name|operator_id|operator_name                                 |monday|tuesday|wednesday|thursday|friday|saturday|sunday|start_date|end_date  |route_short_name|transport_clean|
+------------------------+---------------+-----------------------+---------------+--------------+-------------+------------+--------------+-------------+----------+---------------+-----------+----------------------------------------------+------+-----

26/05/14 14:16:01 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB


In [70]:
trips_df = spark.read.parquet("/user/groups/com-490/H1/final/v1/trip_stop_events.parquet")
trips_df.createOrReplaceTempView("base_trips")

In [71]:
timetable_df = spark.sql("""
WITH date_range AS (
    SELECT explode(sequence(to_date('2026-02-01'), to_date('2026-02-28'))) AS target_date
),
calendar_expanded AS (
    SELECT 
        d.target_date,
        t.*,
        lower(date_format(d.target_date, 'EEEE')) as day_name
    FROM date_range d
    CROSS JOIN base_trips t
    WHERE d.target_date BETWEEN t.start_date AND t.end_date
),
exceptions AS (
    SELECT service_id, exception_date as ex_date, exception_type 
    FROM iceberg.sbb.calendar_dates
    WHERE exception_date BETWEEN '2026-02-01' AND '2026-02-28'
)
SELECT DISTINCT
    -- We calculate the actual calendar date first (handling the >24h GTFS logic)
    -- then cast to a timestamp using the normalized HH:mm:ss
    to_timestamp(
        concat(
            date_add(ce.target_date, CASE WHEN CAST(split(ce.arrival_time, ':')[0] AS INT) >= 24 THEN 1 ELSE 0 END),
            ' ',
            CASE 
                WHEN CAST(split(ce.arrival_time, ':')[0] AS INT) >= 24 
                THEN concat(lpad(CAST(split(ce.arrival_time, ':')[0] AS INT) - 24, 2, '0'), ':', split(ce.arrival_time, ':')[1], ':', split(ce.arrival_time, ':')[2])
                ELSE ce.arrival_time 
            END
        )
    ) AS arrival_timestamp,

    to_timestamp(
        concat(
            date_add(ce.target_date, CASE WHEN CAST(split(ce.departure_time, ':')[0] AS INT) >= 24 THEN 1 ELSE 0 END),
            ' ',
            CASE 
                WHEN CAST(split(ce.departure_time, ':')[0] AS INT) >= 24 
                THEN concat(lpad(CAST(split(ce.departure_time, ':')[0] AS INT) - 24, 2, '0'), ':', split(ce.departure_time, ':')[1], ':', split(ce.departure_time, ':')[2])
                ELSE ce.departure_time 
            END
        )
    ) AS departure_timestamp,

    ce.trip_id,
    ce.stop_id,
    ce.stop_name,
    ce.stop_lat,
    ce.stop_lon,
    ce.stop_sequence,
    ce.service_id,
    ce.trip_short_name,
    ce.operator_id,
    ce.operator_name,
    ce.route_short_name,
    ce.transport_clean,
    CAST(NULL AS TIMESTAMP) AS predicted_arrival_time,
    CAST(NULL AS TIMESTAMP) AS predicted_departure_time

FROM calendar_expanded ce
LEFT JOIN exceptions ex 
    ON ce.service_id = ex.service_id AND ce.target_date = ex.ex_date
WHERE 
    (
        (
            (ce.day_name = 'monday' AND ce.monday = 1) OR
            (ce.day_name = 'tuesday' AND ce.tuesday = 1) OR
            (ce.day_name = 'wednesday' AND ce.wednesday = 1) OR
            (ce.day_name = 'thursday' AND ce.thursday = 1) OR
            (ce.day_name = 'friday' AND ce.friday = 1) OR
            (ce.day_name = 'saturday' AND ce.saturday = 1) OR
            (ce.day_name = 'sunday' AND ce.sunday = 1)
        )
        AND (ex.exception_type IS NULL OR ex.exception_type != 2)
    )
    OR (ex.exception_type = 1)
ORDER BY departure_timestamp
""")

# 3. Save the result
output_timetable_path = "/user/groups/com-490/H1/final/v1/timetable_february.parquet"
timetable_df.write.mode("overwrite").parquet(output_timetable_path)
output_transfers_path = "/user/groups/com-490/H1/final/v1/transfers.parquet"
df_transfers.write.mode("overwrite").parquet(output_transfers_path)

In [72]:
trips_df = spark.read.parquet("/user/groups/com-490/H1/final/v1/timetable_february.parquet")
trips_df.createOrReplaceTempView("february_trips")

In [73]:
spark.sql("SELECT COUNT(*) FROM february_trips").show()

+--------+
|count(1)|
+--------+
| 3071095|
+--------+



In [74]:
spark.sql("SELECT * FROM february_trips LIMIT 1").show(1, truncate=False)

+-------------------+-------------------+------------------------+---------------+---------------+---------------+--------------+-------------+----------+---------------+-----------+----------------------------------------------+----------------+---------------+----------------------+------------------------+
|arrival_timestamp  |departure_timestamp|trip_id                 |stop_id        |stop_name      |stop_lat       |stop_lon      |stop_sequence|service_id|trip_short_name|operator_id|operator_name                                 |route_short_name|transport_clean|predicted_arrival_time|predicted_departure_time|
+-------------------+-------------------+------------------------+---------------+---------------+---------------+--------------+-------------+----------+---------------+-----------+----------------------------------------------+----------------+---------------+----------------------+------------------------+
|2026-02-04 18:31:00|2026-02-04 18:31:00|314.TA.92-42-E-j26-1.2.R|8